# 04 TodoListMiddleware 中间件（待办事项列表管理）

> ✅ **名称修正说明**
> 本文件对应官方中间件 **`TodoListMiddleware`**（`langchain.agents.middleware.todo`）。
> 该中间件为 Agent 提供 **“待办事项列表（Todo List）”管理能力**，
> 是官方中间件中唯一负责“任务规划 / 进度追踪”的组件。

---

## 0. 本文件目录

| 章节 | 内容 |
| --- | --- |
| 1 | 功能简介 |
| 2 | 构造函数参数详解 |
| 3 | 内部机制与执行流程 |
| 4 | 典型使用场景 |
| 5 | 注意事项与易错点 |
| 6 | 示例1：基础用法（复杂多步任务自动建 todo） |
| 7 | 示例2：读取 todos 状态 + 自定义 tool_description |
| 8 | 示例3：自定义 system_prompt（中文规则） |
| 9 | 机制补充：write_todos 工具与“禁止并行”规则 |

---

## 1. 功能简介

**一句话总结**：`TodoListMiddleware` 给 Agent 注册一个 **`write_todos` 工具** 和一套 **系统提示词**，
让 Agent 在处理**复杂多步任务**时，能主动创建、更新、完成一份“待办事项列表”，
并在每次模型调用前自动把使用规范注入到系统消息里。

- ✅ **任务可视化**：用户能通过 `todos` 状态看到 Agent 正在做什么、做到哪一步；
- ✅ **规划更可靠**：Agent 被引导先拆解任务、再逐步执行，减少遗漏；
- ✅ **多步任务不迷路**：执行过程中可随时增删改任务，跟随新信息调整计划。

### 它提供了什么？

| 能力 | 说明 |
| --- | --- |
| `write_todos` 工具 | Agent 可调用的工具，**每次调用整体替换**整个 todo 列表 |
| 系统提示词注入 | 每次模型调用前自动追加“何时用、怎么用”的规范说明 |
| `todos` 状态字段 | Agent 状态中新增 `todos: list[Todo]`，随 Agent 返回 |
| 并行调用防护 | 强制 `write_todos` **每轮最多调用 1 次**（并行调用会被拒绝） |

### Todo 条目结构（TypedDict）

```python
class Todo(TypedDict):
    content: str                                       # 任务内容描述
    status: Literal["pending", "in_progress", "completed"]  # 任务状态
```

| status 取值 | 含义 |
| --- | --- |
| `pending` | 尚未开始 |
| `in_progress` | 正在执行（可同时有多个） |
| `completed` | 已完成 |

---

## 2. 构造函数参数详解

```python
TodoListMiddleware(
    *,
    system_prompt: str = WRITE_TODOS_SYSTEM_PROMPT,
    tool_description: str = WRITE_TODOS_TOOL_DESCRIPTION,
)
```

| 参数 | 类型 | 默认值 | 作用说明 |
| --- | --- | --- | --- |
| `system_prompt` | `str` | 内置英文规范（见源码 `WRITE_TODOS_SYSTEM_PROMPT`） | 每次模型调用前**追加到系统消息末尾**的提示词，指导 Agent 何时用、如何用 `write_todos`。可自定义为中文规则。 |
| `tool_description` | `str` | 内置英文描述（见源码 `WRITE_TODOS_TOOL_DESCRIPTION`） | `write_todos` 工具的 description，影响模型对工具用途的理解。可自定义。 |

> 注意：两个参数都是**关键字参数**，且都**可省略**——不传即可用官方默认行为。

### 内部默认提示词要点（默认 system_prompt 规范）

- ✅ **该用时**：3 步以上的复杂任务、用户明确要求、用户给出多任务清单；
- ❌ **不该用**：单一直接任务、纯闲聊、少于 3 步的琐碎任务；
- 🔁 执行中实时更新：开始任务标记 `in_progress`，完成立刻标记 `completed`；
- 🚫 遇到阻塞时**不要**标记完成，而是新增一条“待解决”任务；
- ⚠️ `write_todos` **不允许并行调用**（每次调用会整体替换列表，并行会语义冲突）。

---

## 3. 内部机制与执行流程

`TodoListMiddleware` 同时实现了 **钩子型** 与 **包装型** 两类机制：

### ① wrap_model_call —— 每次模型调用前注入系统提示词（包装型）

```
模型调用 → [TodoListMiddleware.wrap_model_call]
             ├─ 读取 request.system_message
             ├─ 若已有系统消息：在其 content_blocks 后追加 "\n\n{self.system_prompt}"
             ├─ 若没有系统消息：新建 SystemMessage(仅含 self.system_prompt)
             └─ request.override(system_message=新消息) → 继续调用 handler
```

效果：**无需修改 create_agent 的 system_prompt**，Agent 自动获得 todo 使用规范。

### ② 注册 write_todos 工具（工具提供型）

```python
@tool(description=self.tool_description)
def write_todos(todos: list[Todo], tool_call_id=InjectedToolCallId) -> Command[Any]:
    return Command(update={
        "todos": todos,                                              # 整体替换 todo 列表
        "messages": [ToolMessage(f"Updated todo list to {todos}", tool_call_id=...)],  # 回执
    })
```

- 工具通过 `middleware.tools` 自动注册进 Agent（`create_agent(tools=...)` 不用额外加）；
- 每次调用**整体替换** todo 列表（不是增量更新）——这是“禁止并行调用”的根源；
- 返回 `Command(update=...)` 直接写入 Agent 状态。

### ③ after_model —— 并行调用防护（钩子型）

```
模型输出后 → [TodoListMiddleware.after_model]
              ├─ 取最后一条 AI 消息
              ├─ 统计其中 write_todos 的调用次数
              ├─ 若 > 1：为每个 write_todos 调用注入 status="error" 的 ToolMessage
              │         （提示“write_todos 不应被并行调用，请每轮只调用一次”）
              └─ 否则：返回 None，正常执行
```

### ④ state_schema —— 扩展 Agent 状态

`state_schema = PlanningState`，在原有 Agent 状态上新增：

```python
todos: Annotated[NotRequired[list[Todo]], OmitFromInput]  # 用户无需输入
```

任务结束时，`todos` 会随 Agent 返回值一起返回，可直接读取。

---

## 4. 典型使用场景

1. **复杂多步任务**：如“调研 5 个竞品并输出对比报告”——Agent 先建 todo，再逐项完成；
2. **需要让用户看到进度**：长任务执行过程中，用户随时可见“进行到第几步”；
3. **需要中途调整计划**：执行中发现新任务/任务作废，Agent 动态增删 todo 项；
4. **与其它中间件组合**：如配合 `LLMToolSelectorMiddleware` 裁剪工具、
   `ModelCallLimitMiddleware` 限制调用次数，构建生产级 Agent（见 06 文件）。

---

## 5. 注意事项与易错点

- ⚠️ **`write_todos` 每次调用会整体替换 todo 列表**：并行调用会产生“谁优先”的歧义，
  因此 `after_model` 会拒绝**同一轮里多个** `write_todos` 调用。
- ⚠️ **简单任务不会用 todo**：默认提示词明确要求“少于 3 步的琐碎任务直接用，不调工具”，
  所以演示时请给 Agent 一个**多步复杂任务**才能看到效果。
- 💡 `todos` 是**状态字段**：读取方式为 `result["todos"]`（Agent 返回的 state dict）。
- 💡 工具名是固定的 `write_todos`：与其它工具组合时注意不要重名。
- 💡 自定义 `system_prompt` 完全替换默认规范，若只改一处用法请保留其余规则文本。
- 💡 该中间件**不限制任何外部工具**，只做“规划 + 状态管理”。

---

In [1]:
# ============================================================
# 环境准备：加载 .env 中的 API Key，并初始化主模型
# ============================================================
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from rich import print as rprint

load_dotenv(override=True)

llm = init_chat_model(model="deepseek:deepseek-v4-flash",
                      extra_body={
                          "thinking":{"type":"disabled"}
                      })
print("主模型初始化完成：", type(llm).__name__)

主模型初始化完成： ChatDeepSeek


In [2]:
# ============================================================
# 定义两个配套工具，凑出一个“多步复杂任务”场景
# ============================================================
@tool
def get_weather(city: str) -> str:
    """查询指定城市当前的天气情况。"""
    return f"{city}：晴，26℃，微风，空气质量良"


@tool
def calculator(expression: str) -> str:
    """计算数学表达式并返回结果，例如 '1 + 2 * 3'。"""
    # 注意：eval 仅用于教学演示，生产环境请勿直接 eval 用户输入！
    try:
        return f"{expression} = {eval(expression)}"
    except Exception as e:  # noqa: BLE001
        return f"计算失败：{e}"


print("工具定义完成")

工具定义完成


In [5]:
# ============================================================
# 示例1：基础用法 —— 复杂多步任务自动规划
#
# 任务设计为“多步复杂任务”（查两个城市天气 + 计算预算 + 给建议），
# 触发默认提示词规则，让 Agent 主动调用 write_todos 建清单。
# ============================================================

# 直接用默认参数创建中间件
todo_mw = TodoListMiddleware(
    system_prompt="""
你是一个擅长规划和执行复杂任务的 Agent。

必须遵守以下规则：

1. 当任务包含两个或以上独立步骤时，
   必须先调用 write_todos 创建任务清单。
2. 第一次调用 write_todos 时，
   将所有主要任务拆解为清晰的待办事项。
3. 初始清单中，第一个即将执行的任务标记为 in_progress，
   其余任务标记为 pending。
4. 创建清单时，只调用 write_todos，
   不要在同一轮同时调用天气、计算器等业务工具。
5. 收到 write_todos 的工具回执后，
   再调用业务工具完成任务。
6. 每完成一个任务，及时调用 write_todos 更新状态；
   每次调用都要传入完整的最新清单。
7. 所有任务完成后，将对应任务标记为 completed。
8. 不得将未完成的任务标记为 completed。
9. write_todos 每轮最多调用一次，不允许并行调用。
"""
)

agent = create_agent(
    model=llm,
    tools=[get_weather, calculator],
    middleware=[todo_mw],          # 挂上 todo 中间件
)

resp = await agent.ainvoke({
    "messages": [
        HumanMessage(
            content="我周末想从北京去上海出差，请帮我：\n"
                    "1. 查询北京和上海两地的天气；\n"
                    "2. 用计算器估算高铁往返路费 1100 元 + 酒店两晚 800 元/晚 的总预算；\n"
                    "3. 结合天气给出穿衣和出行建议。"
        )
    ]
})


rprint(resp)

# ============================================================
# 观察点1：Agent 状态中的 todos —— 任务列表及状态流转
# ============================================================
print("\n=== todos 状态 ===")
for todo in resp.get("todos", []):
    rprint(f"[{todo['status']}] {todo['content']}")

# ============================================================
# 观察点2：完整消息流 —— 能看到 write_todos 的调用与回执
# ============================================================
print("\n=== 完整消息流 ===")
for msg in resp["messages"]:
    msg.pretty_print()

{
    'messages': [
        HumanMessage(
            content='我周末想从北京去上海出差，请帮我：\n1. 查询北京和上海两地的天气；\n2. 用计算器估算高铁往返路费
1100 元 + 酒店两晚 800 元/晚 的总预算；\n3. 结合天气给出穿衣和出行建议。',
            additional_kwargs={},
            response_metadata={},
            id='f09edfba-339d-404b-8c0d-333152cf07ae'
        ),
        AIMessage(
            content="I'll create a task list first, then work through the steps.",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 132,
                    'prompt_tokens': 1571,
                    'total_tokens': 1703,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 1571
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '1b14d769-d23a-45a4-b2d5-96e440d77207',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a131f8-9521-7a52-b097-91a9ba1f159a-0',
            tool_calls=[
                {
                    'name': 'write_todos',
                    'args': {
                        'todos': [
                            {'content': '查询北京当前天气', 'status': 'in_progress'},
                            {'content': '查询上海当前天气', 'status': 'pending'},
                            {'content': '用计算器计算总预算（高铁往返1100 + 酒店800×2）', 'status': 'pending'},
                            {'content': '结合天气给出穿衣和出行建议', 'status': 'pending'}
                        ]
                    },
                    'id': 'call_00_kG1uoUTMfUx0D0TLouwe6576',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1571,
                'output_tokens': 132,
                'total_tokens': 1703,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Updated todo list to [{'content': '查询北京当前天气', 'status': 'in_progress'}, {'content': 
'查询上海当前天气', 'status': 'pending'}, {'content': '用计算器计算总预算（高铁往返1100 + 酒店800×2）', 'status': 
'pending'}, {'content': '结合天气给出穿衣和出行建议', 'status': 'pending'}]",
            name='write_todos',
            id='71af017e-72dc-4676-8b38-d218e585e79a',
            tool_call_id='call_00_kG1uoUTMfUx0D0TLouwe6576'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 67,
                    'prompt_tokens': 1803,
                    'total_tokens': 1870,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1664},
                    'prompt_cache_hit_tokens': 1664,
                    'prompt_cache_miss_tokens': 139
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '90c0b45c-d93f-4a86-85ba-837a131e545e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a131f8-992e-7213-ad65-a83c451a8c66-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_kpQs7zibGHoUlgmVtuLx0510',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_weather',
                    'args': {'city'


=== todos 状态 ===


查询北京当前天气

查询上海当前天气

用计算器计算总预算（高铁往返1100 + 酒店800×2）

结合天气给出穿衣和出行建议


=== 完整消息流 ===
================================ Human Message =================================

我周末想从北京去上海出差，请帮我：
1. 查询北京和上海两地的天气；
2. 用计算器估算高铁往返路费 1100 元 + 酒店两晚 800 元/晚 的总预算；
3. 结合天气给出穿衣和出行建议。
================================== Ai Message ==================================

I'll create a task list first, then work through the steps.
Tool Calls:
  write_todos (call_00_kG1uoUTMfUx0D0TLouwe6576)
 Call ID: call_00_kG1uoUTMfUx0D0TLouwe6576
  Args:
    todos: [{'content': '查询北京当前天气', 'status': 'in_progress'}, {'content': '查询上海当前天气', 'status': 'pending'}, {'content': '用计算器计算总预算（高铁往返1100 + 酒店800×2）', 'status': 'pending'}, {'content': '结合天气给出穿衣和出行建议', 'status': 'pending'}]
================================= Tool Message =================================
Name: write_todos

Updated todo list to [{'content': '查询北京当前天气', 'status': 'in_progress'}, {'content': '查询上海当前天气', 'status': 'pending'}, {'content': '用计算器计算总预算（高铁往返1100 + 酒店800×2）', 'status': 'pending'}, {'content': '结合天气给出穿衣和出行建议', 'status': 'p

In [6]:
# ============================================================
# 示例2：读取 todos 状态字段 + 自定义 tool_description
#
# 1. write_todos 工具由中间件自动注册（可通过 middleware.tools 查看）；
# 2. 自定义 tool_description 让模型更贴合业务语义；
# 3. todos 以状态字段形式随 Agent 返回。
# ============================================================

# 自定义工具描述：把“写待办”改造成更符合中文业务的描述
todo_mw2 = TodoListMiddleware(
    tool_description="创建并维护一份『任务清单』：用于多步任务的进度追踪。"
                     "注意：每次调用会用新清单整体替换旧清单，且一轮只能调用一次。",
)

# 确认 write_todos 已自动注册到中间件的 tools 里
rprint("中间件自动注册的工具：", [t.name for t in todo_mw2.tools])

agent2 = create_agent(
    model=llm,
    tools=[get_weather, calculator],
    middleware=[todo_mw2],
)

resp2 = await agent2.ainvoke({
    "messages": [
        HumanMessage(content="请分三步完成：①查北京天气 ②查上海天气 ③计算两地温差并告诉我")
    ]
})

print("\n=== todos 状态 ===")
for todo in resp2.get("todos", []):
    rprint(f"[{todo['status']}] {todo['content']}")

print("\n=== 最终回答 ===")
resp2["messages"][-1].pretty_print()

中间件自动注册的工具：
['write_todos']


=== todos 状态 ===

=== 最终回答 ===
================================== Ai Message ==================================

三步都已完成，结果如下：

**① 北京天气**：晴，26℃，微风，空气质量良
**② 上海天气**：晴，26℃，微风，空气质量良
**③ 两地温差**：26℃ − 26℃ = **0℃**

两地当前温度完全相同，没有温差。另外两地的天气状况（晴、微风、空气质量良）也完全一致——如果你需要判断这点是否异常、或者想换个数据源核对，可以告诉我。


In [7]:
# ============================================================
# 示例3：自定义 system_prompt —— 用中文规则约束 Agent 的规划行为
#
# 通过自定义 system_prompt，可以完全替换默认的英文规范。
# 例如：强制 Agent“任何任务都要先建清单”，或“禁止使用 todo”。
# ============================================================

todo_mw3 = TodoListMiddleware(
    system_prompt=(
        "## 任务清单使用规则（自定义）\n"
        "你拥有 write_todos 工具用于管理任务清单。\n"
        "规则：\n"
        "1. 收到任何任务，第一步先调用 write_todos 建立清单，并立即把第一项标记为 in_progress；\n"
        "2. 每完成一项，立即将其标记为 completed，并把下一项标记为 in_progress；\n"
        "3. 每轮只能调用一次 write_todos，禁止并行调用；\n"
        "4. 所有清单内容使用中文描述。"
    ),
)

agent3 = create_agent(
    model=llm,
    tools=[get_weather, calculator],
    middleware=[todo_mw3],
)

resp3 = await agent3.ainvoke({
    "messages": [
        HumanMessage(content="帮我查询北京的天气，然后计算一下 15% 的增值税对应 2600 元是多少")
    ]
})

print("\n=== todos 状态 ===")
for todo in resp3.get("todos", []):
    rprint(f"[{todo['status']}] {todo['content']}")

print("\n=== 完整消息流（观察 write_todos 调用） ===")
for msg in resp3["messages"]:
    msg.pretty_print()


=== todos 状态 ===


查询北京的天气

计算 2600 元的 15% 增值税


=== 完整消息流（观察 write_todos 调用） ===
================================ Human Message =================================

帮我查询北京的天气，然后计算一下 15% 的增值税对应 2600 元是多少
================================== Ai Message ==================================

I'll help you with both tasks. Let me start by creating a task list.
Tool Calls:
  write_todos (call_00_hEpTzOWGGAYuySs9Ex5u7936)
 Call ID: call_00_hEpTzOWGGAYuySs9Ex5u7936
  Args:
    todos: [{'content': '查询北京的天气', 'status': 'in_progress'}, {'content': '计算 2600 元的 15% 增值税', 'status': 'pending'}]
================================= Tool Message =================================
Name: write_todos

Updated todo list to [{'content': '查询北京的天气', 'status': 'in_progress'}, {'content': '计算 2600 元的 15% 增值税', 'status': 'pending'}]
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_C8O8l2ePxJL6DTpzr7F49781)
 Call ID: call_00_C8O8l2ePxJL6DTpzr7F49781
  Args:
    city: 北京
  calculator (call_01_pXZNjIA1BDq

## 9. 机制补充：write_todos 工具与“禁止并行”规则

### ① 为什么禁止并行调用 `write_todos`？

因为 `write_todos` 的语义是 **“用本次传入的列表整体替换旧列表”**。
如果同一轮模型输出里同时发起 2 次 `write_todos` 调用，谁先谁后没有定义，
两个结果互相覆盖，状态不可预期。因此 `after_model` 会拦截这种场景：

```python
# after_model 中的关键逻辑（源码摘录）
if len(write_todos_calls) > 1:
    error_messages = [
        ToolMessage(
            content="Error: The `write_todos` tool should never be called multiple times "
                    "in parallel. Please call it only once per model invocation...",
            tool_call_id=tc["id"],
            status="error",
        )
        for tc in write_todos_calls
    ]
    return {"messages": error_messages}   # 注入错误消息，Agent 会看到并修正
```

> 若想故意观察该拦截效果，可在自定义 `system_prompt` 中要求模型“同时写两份清单”，
> 或使用支持并行工具调用的模型输出多个 `write_todos` —— 此时每条都会被替换为 error 回执。

### ② write_todos 返回什么？

工具返回 `Command(update={...})`，直接写入 Agent 状态：
- `todos`：新的完整任务列表；
- `messages`：一条 `ToolMessage("Updated todo list to ...")` 回执，供模型确认。

### ③ 状态字段 `todos` 如何读取？

```python
result = await agent.ainvoke({...})
result["todos"]   # list[Todo]，如 [{"content": "...", "status": "in_progress"}, ...]
```

> 本文件完。核心结论：**`TodoListMiddleware` = `write_todos` 工具 + 系统提示词注入 +
> `todos` 状态字段 + 并行调用防护**，一句话即可启用：
> `create_agent(model=..., middleware=[TodoListMiddleware()])`。